# AI Surveillance - Automatic License Plate Recognition (ALPR) Demo

This notebook demonstrates **Vehicle Detection and Automatic License Plate Recognition (ALPR)**. Using standard YOLOv11 to track vehicles (cars, trucks, motorcycles, buses), a pre-trained license plate detection model from Hugging Face (`Koushim/yolov8-license-plate-detection`), and `easyocr` for optical character recognition, it reads license numbers dynamically.

### Google Colab Instructions:
1. Click the "Runtime" menu option and select **"Change runtime type"**. Make sure **T4 GPU** is selected for optimal performance.
2. Run the cells sequentially to install dependencies, run inference, and visualize results.

## 1. Install Dependencies


In [ ]:
# Install dependencies (ultralytics, easyocr, opencv, matplotlib, numpy)
!pip install ultralytics opencv-python matplotlib numpy easyocr


## 2. Import Libraries


In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import easyocr
import os
import urllib.request
%matplotlib inline


## 3. Load AI Core Models


In [ ]:
print("Loading YOLOv11 Vehicle Tracking model...")
vehicle_model = YOLO("yolo11n.pt")

print("Downloading License Plate Localization model weights from Hugging Face Hub...")
# Standard YOLO class doesn't support custom HF repo identifiers natively.
# We download the 'best.pt' weights manually to bypass loader errors and ensure a robust Colab experience.
plate_model_url = "https://huggingface.co/Koushim/yolov8-license-plate-detection/resolve/main/best.pt"
plate_model_path = "license_plate_detector.pt"
if not os.path.exists(plate_model_path):
    print("Downloading plate detection model weight file...")
    urllib.request.urlretrieve(plate_model_url, plate_model_path)
    print("Model weights download completed successfully!")
else:
    print("Model weights already present locally.")

print("Loading License Plate Localization model...")
plate_model = YOLO(plate_model_path)

print("Initializing EasyOCR Engine (seeking GPU)...")
reader = easyocr.Reader(['en'], gpu=True)
print("Models and OCR loaded successfully!")


## 4. Load and Recognize License Plate on a Test Image


In [ ]:
# Download a sample vehicle with a highly visible license plate using requests to bypass 403 Forbidden errors
import requests
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b2/UK_license_plate_format.jpg/640px-UK_license_plate_format.jpg"
image_path = "test_car.jpg"
response = requests.get(image_url, headers={'User-Agent': 'Mozilla/5.0'})
with open(image_path, 'wb') as f:
    f.write(response.content)

# Read image and run models
img = cv2.imread(image_path)
annotated_img = img.copy()

# 1. Run plate detection
plate_results = plate_model(img, verbose=False)

print("Running OCR on localized license plates...")

if plate_results and len(plate_results) > 0:
    boxes = plate_results[0].boxes
    for i, box in enumerate(boxes):
        xyxy = box.xyxy[0].cpu().numpy()
        px1, py1, px2, py2 = map(int, xyxy)
        conf = float(box.conf[0].item())
        
        # Draw plate boundary
        cv2.rectangle(annotated_img, (px1, py1), (px2, py2), (0, 255, 0), 3)
        
        # Crop the plate with a small margin padding for OCR improvement
        h, w = img.shape[:2]
        pad_x = int((px2 - px1) * 0.05)
        pad_y = int((py2 - py1) * 0.05)
        
        crop_x1 = max(0, px1 - pad_x)
        crop_y1 = max(0, py1 - pad_y)
        crop_x2 = min(w, px2 + pad_x)
        crop_y2 = min(h, py2 + pad_y)
        
        plate_crop = img[crop_y1:crop_y2, crop_x1:crop_x2]
        
        # Perform OCR
        ocr_res = reader.readtext(plate_crop)
        text = "UNKNOWN"
        ocr_conf = 0.0
        
        if ocr_res:
            # Sort OCR results by confidence, pick the strongest text match
            ocr_res = sorted(ocr_res, key=lambda x: x[2], reverse=True)
            text = ocr_res[0][1].upper().strip().replace(" ", "")
            ocr_conf = ocr_res[0][2]
            
        print(f"License Plate Detected! Conf: {conf*100:.1f}% | OCR: '{text}' (Conf: {ocr_conf*100:.1f}%)")
        
        # Overlay reading on image
        label = f"PLATE: {text} ({int(ocr_conf*100)}%)"
        cv2.putText(annotated_img, label, (px1, px1 - 10 if px1 - 10 > 15 else py1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

# Show visual result
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
plt.title("ALPR Static Inference Result")
plt.axis('off')
plt.show()


## 5. Download Test Video


In [ ]:
# We download a highway sample video with moving vehicles
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4"
video_input_path = "highway_input.mp4"
video_output_path = "highway_output_raw.mp4"
video_final_path = "highway_output.mp4"

if not os.path.exists(video_input_path):
    print("Downloading sample highway video...")
    urllib.request.urlretrieve(video_url, video_input_path)
    print("Download completed!")
else:
    print("Sample video already exists.")


## 6. Run Vehicle and Plate Tracker Pipeline on Video File


In [ ]:
cap = cv2.VideoCapture(video_input_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_output_path, fourcc, fps, (width, height))

# Classes corresponding to vehicles in COCO dataset: 2 (car), 3 (motorcycle), 5 (bus), 7 (truck)
vehicle_classes = [2, 3, 5, 7]
print("Processing highway video stream (Vehicle + License Plate Detection)...")
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    frame_idx += 1
    if frame_idx % 30 == 0 or frame_idx == 1:
        print(f"Processing frame {frame_idx}/{total_frames}...")
        
    annotated_frame = frame.copy()
    
    # 1. Track Vehicles
    veh_results = vehicle_model.track(source=frame, persist=True, conf=0.3, classes=vehicle_classes, imgsz=320, verbose=False)
    
    if veh_results and len(veh_results) > 0 and veh_results[0].boxes is not None:
        boxes = veh_results[0].boxes
        for i in range(len(boxes)):
            box = boxes[i]
            track_id = int(box.id[0].item()) if box.id is not None else 1
            class_id = int(box.cls[0].item())
            class_name = vehicle_model.names[class_id].upper()
            
            vx1, vy1, vx2, vy2 = map(int, box.xyxy[0].cpu().numpy())
            
            # Draw Vehicle box
            cv2.rectangle(annotated_frame, (vx1, vy1), (vx2, vy2), (255, 0, 0), 2)
            
            # Crop Vehicle to look for license plate
            veh_crop = frame[vy1:vy2, vx1:vx2]
            if veh_crop.size == 0: continue
            
            # 2. Detect Plate inside vehicle crop
            plate_results = plate_model(veh_crop, conf=0.2, verbose=False)
            plate_text = ""
            
            if plate_results and len(plate_results) > 0 and plate_results[0].boxes is not None:
                p_boxes = plate_results[0].boxes
                for pb in p_boxes:
                    px1, py1, px2, py2 = map(int, pb.xyxy[0].cpu().numpy())
                    
                    # Calculate global plate coordinates
                    g_px1, g_py1 = vx1 + px1, vy1 + py1
                    g_px2, g_py2 = vx1 + px2, vy1 + py2
                    
                    cv2.rectangle(annotated_frame, (g_px1, g_py1), (g_px2, g_py2), (0, 255, 0), 2)
                    
                    # Crop plate for OCR (only every 3rd frame to optimize speed)
                    if frame_idx % 3 == 0 or frame_idx == 1:
                        plate_crop = veh_crop[py1:py2, px1:px2]
                        if plate_crop.size > 0:
                            ocr_res = reader.readtext(plate_crop)
                            if ocr_res:
                                plate_text = sorted(ocr_res, key=lambda x: x[2], reverse=True)[0][1]
                                plate_text = plate_text.upper().strip().replace(" ", "")
            
            # Display vehicle labels
            label = f"{class_name} #{track_id}"
            if plate_text:
                label += f" [PLATE: {plate_text}]"
            cv2.putText(annotated_frame, label, (vx1, vy1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
            
    out.write(annotated_frame)
    
cap.release()
out.release()
print("Video processing complete!")


## 7. Convert Output Video to H.264


In [ ]:
print("Converting video format...")
os.system(f"ffmpeg -y -i {video_output_path} -vcodec libx264 -f mp4 {video_final_path}")
print("Format conversion complete!")


## 8. Play Transcoded Video Inline


In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open(video_final_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<div style="text-align: center;">
    <h3 style="color: #27ae60; font-family: sans-serif;">Automatic License Plate Recognition Stream</h3>
    <video width="640" height="480" controls style="border: 2px solid #27ae60; border-radius: 12px; box-shadow: 0 5px 15px rgba(0,0,0,0.3);">
        <source src="{data_url}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
</div>
""")


## 9. Setup Colab Webcam Bridge Utilities


In [ ]:
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
import PIL.Image

def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

def bbox_to_bytes(bbox_array):
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))
    return bbox_bytes


## 10. Initialize Web Camera Stream CSS/JS Component


In [ ]:
def start_webcam_stream():
    js_code = """
    var video;
    var container;
    var canvas;
    var labelVal;
    var plateTable;
    var seenPlates = new Set();
    
    async function initWebcam() {
      container = document.createElement('div');
      container.style.display = 'flex';
      container.style.flexDirection = 'column';
      container.style.alignItems = 'center';
      container.style.fontFamily = 'system-ui, -apple-system, sans-serif';
      container.style.background = '#0f172a';
      container.style.padding = '20px';
      container.style.borderRadius = '16px';
      container.style.boxShadow = '0 10px 30px rgba(0,0,0,0.5)';
      container.style.width = '680px';
      container.style.margin = '20px auto';
      container.style.color = '#ffffff';
      container.style.border = '2px solid #1e293b';
      
      const title = document.createElement('h3');
      title.innerText = 'Surveillance ALPR Intelligence Module';
      title.style.margin = '0 0 15px 0';
      title.style.color = '#38bdf8';
      title.style.fontSize = '1.3rem';
      container.appendChild(title);
      
      const videoWrapper = document.createElement('div');
      videoWrapper.style.position = 'relative';
      videoWrapper.style.width = '640px';
      videoWrapper.style.height = '480px';
      videoWrapper.style.borderRadius = '10px';
      videoWrapper.style.overflow = 'hidden';
      videoWrapper.style.border = '2px solid #334155';
      
      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 640;
      video.height = 480;
      videoWrapper.appendChild(video);
      
      canvas = document.createElement('canvas');
      canvas.style.position = 'absolute';
      canvas.style.top = '0';
      canvas.style.left = '0';
      canvas.width = 640;
      canvas.height = 480;
      videoWrapper.appendChild(canvas);
      container.appendChild(videoWrapper);
      
      labelVal = document.createElement('div');
      labelVal.innerText = 'Scanning for license plates...';
      labelVal.style.marginTop = '15px';
      labelVal.style.fontSize = '1rem';
      labelVal.style.fontWeight = 'bold';
      labelVal.style.color = '#38bdf8';
      container.appendChild(labelVal);
      
      // Plate database table
      const tableTitle = document.createElement('div');
      tableTitle.innerText = 'RECOGNIZED VEHICLES & PLATES TELEMETRY';
      tableTitle.style.marginTop = '15px';
      tableTitle.style.fontSize = '0.8rem';
      tableTitle.style.letterSpacing = '0.1em';
      tableTitle.style.color = '#94a3b8';
      container.appendChild(tableTitle);
      
      plateTable = document.createElement('table');
      plateTable.style.width = '100%';
      plateTable.style.marginTop = '8px';
      plateTable.style.borderCollapse = 'collapse';
      plateTable.style.background = '#1e293b';
      plateTable.style.borderRadius = '8px';
      plateTable.style.fontSize = '0.85rem';
      plateTable.innerHTML = '<thead><tr style="border-bottom: 2px solid #334155;"><th style="padding: 8px; text-align: left;">Timestamp</th><th style="padding: 8px; text-align: left;">Plate Code</th><th style="padding: 8px; text-align: left;">Alert Class</th></tr></thead><tbody id="telemetry-body"></tbody>';
      container.appendChild(plateTable);
      
      const stopBtn = document.createElement('button');
      stopBtn.innerText = 'HALT VEHICLE SURVEILLANCE';
      stopBtn.style.marginTop = '15px';
      stopBtn.style.padding = '12px 24px';
      stopBtn.style.background = '#ef4444';
      stopBtn.style.color = 'white';
      stopBtn.style.border = 'none';
      stopBtn.style.borderRadius = '8px';
      stopBtn.style.cursor = 'pointer';
      stopBtn.style.fontWeight = 'bold';
      stopBtn.style.letterSpacing = '0.05em';
      stopBtn.style.transition = 'all 0.2s';
      stopBtn.onmouseover = () => stopBtn.style.background = '#dc2626';
      stopBtn.onmouseout = () => stopBtn.style.background = '#ef4444';
      
      stopBtn.onclick = () => {
        window.webcamActive = false;
        labelVal.innerText = 'Vehicle Surveillance Stopped.';
        labelVal.style.color = '#ef4444';
        const tracks = video.srcObject.getTracks();
        tracks.forEach(track => track.stop());
        video.srcObject = null;
        stopBtn.disabled = true;
        stopBtn.style.background = '#334155';
        stopBtn.style.cursor = 'not-allowed';
      };
      container.appendChild(stopBtn);
      
      document.body.appendChild(container);
      
      const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 } });
      video.srcObject = stream;
      await video.play();
      
      window.webcamActive = true;
    }
    
    async function captureFrame() {
      if (!window.webcamActive) return null;
      const tempCanvas = document.createElement('canvas');
      tempCanvas.width = 640;
      tempCanvas.height = 480;
      tempCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
      return tempCanvas.toDataURL('image/jpeg', 0.8);
    }
    
    function drawOverlay(base64Png, recognizedPlates) {
      if (!window.webcamActive) return;
      const ctx = canvas.getContext('2d');
      ctx.clearRect(0, 0, 640, 480);
      
      if (base64Png) {
        const img = new Image();
        img.onload = function() {
          ctx.drawImage(img, 0, 0);
        };
        img.src = base64Png;
      }
      
      if (recognizedPlates && recognizedPlates.length > 0) {
        const tbody = document.getElementById('telemetry-body');
        recognizedPlates.forEach(plate => {
          if (!seenPlates.has(plate)) {
            seenPlates.add(plate);
            const row = document.createElement('tr');
            row.style.borderBottom = '1px solid #334155';
            row.innerHTML = `<td style="padding: 8px;">${new Date().toLocaleTimeString()}</td><td style="padding: 8px; font-weight: bold; color: #38bdf8;">${plate}</td><td style="padding: 8px; color: #10b981;">DETECTED</td>`;
            tbody.insertBefore(row, tbody.firstChild);
          }
        });
        labelVal.innerText = `Recognized Plates: ${[...seenPlates].slice(-3).join(', ')}`;
      }
    }
    """
    display(Javascript(js_code))


## 11. Run Real-Time Webcam ALPR Stream Loop


In [ ]:
start_webcam_stream()
eval_js('initWebcam()')

vehicle_classes = [2, 3, 5, 7]
frame_idx = 0

try:
    while True:
        js_reply = eval_js('captureFrame()')
        if js_reply is None:
            break
            
        frame = js_to_image(js_reply)
        frame_idx += 1
        
        # Track vehicles
        veh_results = vehicle_model.track(source=frame, persist=True, conf=0.3, classes=vehicle_classes, imgsz=320, verbose=False)
        
        overlay_array = np.zeros((480, 640, 4), dtype=np.uint8)
        found_plates = []
        
        if veh_results and len(veh_results) > 0 and veh_results[0].boxes is not None:
            boxes = veh_results[0].boxes
            for i in range(len(boxes)):
                box = boxes[i]
                track_id = int(box.id[0].item()) if box.id is not None else 1
                class_id = int(box.cls[0].item())
                class_name = vehicle_model.names[class_id].upper()
                
                vx1, vy1, vx2, vy2 = map(int, box.xyxy[0].cpu().numpy())
                
                # Vehicle box (Cyan/Blue)
                cv2.rectangle(overlay_array, (vx1, vy1), (vx2, vy2), (248, 189, 56, 255), 2)
                label = f"{class_name} #{track_id}"
                (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
                cv2.rectangle(overlay_array, (vx1, vy1 - text_h - 6), (vx1 + text_w + 10, vy1), (248, 189, 56, 255), -1)
                cv2.putText(overlay_array, label, (vx1 + 5, vy1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0, 255), 1)

        # Track Plates Independently on Full Frame
        plate_results = plate_model(frame, conf=0.2, verbose=False)
        
        if plate_results and len(plate_results) > 0 and plate_results[0].boxes is not None:
            p_boxes = plate_results[0].boxes
            for pb in p_boxes:
                px1, py1, px2, py2 = map(int, pb.xyxy[0].cpu().numpy())
                
                # Draw plate (Green)
                cv2.rectangle(overlay_array, (px1, py1), (px2, py2), (0, 255, 0, 255), 2)
                
                plate_text = ""
                # OCR every 5 frames
                if frame_idx % 5 == 0 or frame_idx == 1:
                    plate_crop = frame[py1:py2, px1:px2]
                    if plate_crop.size > 0:
                        ocr_res = reader.readtext(plate_crop)
                        if ocr_res:
                            plate_text = sorted(ocr_res, key=lambda x: x[2], reverse=True)[0][1]
                            plate_text = plate_text.upper().strip().replace(" ", "")
                            found_plates.append(plate_text)
                
                if plate_text:
                    label = f"PLATE: {plate_text}"
                    (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
                    cv2.rectangle(overlay_array, (px1, py1 - text_h - 6), (px1 + text_w + 10, py1), (0, 255, 0, 255), -1)
                    cv2.putText(overlay_array, label, (px1 + 5, py1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0, 255), 1)
                
        bbox_bytes = bbox_to_bytes(overlay_array)
        eval_js(f'drawOverlay("{bbox_bytes}", {json.dumps(found_plates)})')
except Exception as e:
    print(f"Stream stopped: {e}")
